# Fine-tuning de LLM

## 1. Préparation

**Cette partie est à exécuter en premier, de façon à télécharger les
bibliothèques, les données et les modèles nécessaires pour la suite du
TP**.

### Installation des bibliothèques (choisir l’une des options)

**Option 1** (solution propre mais un peu plus complexe, passez à
l’option 2 si vous ne comprenez rien)

1.  Si vous n’avez jamais utilisé `uv`, suivez les instructions
    d’installation ici
    <https://docs.astral.sh/uv/getting-started/installation/> (dans
    votre terminal, pas dans le notebook).
2.  Télécharger le fichier
    [`pyproject.toml`](https://schwander.isir.upmc.fr/enseignement/m2probafi_apprentissage/pyproject.toml)
    sur la page du cours.
3.  Mettre à jour l’environnement Python avec `uv sync` (dans votre
    terminal, pas dans le notebook).
4.  Fermer ce notebook et relancez Jupyter avec la commande
    `uv run jupyter notebook` (dans votre terminal, attention à bien
    être dans le répertoire avec le fichier `pyproject.toml`).

**Option 2** (à utiliser notamment avec Google Colab)

Copier-coller ce qui suit dans une cellule de ce notebook et exécuter
cette cellule:

    !pip install --index-url https://download.pytorch.org/whl/cpu "torch>=2.9.0"
    !pip install --index-url https://download.pytorch.org/whl/cpu "torchaudio>=2.9.0"
    !pip install --index-url https://download.pytorch.org/whl/cpu "torchvision>=0.24.0"
    !pip install "bitsandbytes>=0.48.2"
    !pip install "datasets>=4.4.1"
    !pip install "evaluate>=0.4.6"
    !pip install "gensim>=4.4.0"
    !pip install "jupyter>=1.1.1"
    !pip install "mlflow>=3.6.0"
    !pip install "nltk>=3.9.2"
    !pip install "peft>=0.18.0"
    !pip install "polars>=1.35.1"
    !pip install "rouge-score>=0.1.2"
    !pip install "transformers>=4.57.3"
    !pip install "trl>=0.25.1"

### Téléchargements des modèles et des datasets (exécuter ce qui suit)

In [1]:
import torch

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, TrainerCallback
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
import evaluate

import mlflow

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
import os
# Load API key from environment instead of hardcoding secrets.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
if not os.environ["OPENAI_API_KEY"]:
    print("OPENAI_API_KEY is not set; API-based experiments will be skipped.")

Chargement des données:

In [3]:
dataset_name = "medalpaca/medical_meadow_medical_flashcards"
dataset = load_dataset(dataset_name)
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 33955
    })
})

In [4]:
split_dataset = dataset["train"].train_test_split(test_size=0.2)
train_dataset, eval_dataset = split_dataset["train"], split_dataset["test"]

In [5]:
# Juste 100 exemples pour survivre quand on est sur CPU
train_dataset = train_dataset.select(range(80))
eval_dataset = eval_dataset.select(range(20))

Chargement du modèle de base:

In [6]:
model_name = "HuggingFaceTB/SmolLM-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

## 2. Motivation

On s’intéresse ici au fine-tuning d’un petit LLM (135M de paramètres),
dont la petite taille la qualité des réponses, en particulier sur des
domaines spécialisés comme la médecine. Néanmoins, en l’entraînant
spécifiquement sur un corpus adapté on espère améliorer ses performances
sur des questions médicales. L’objectif est d’approcher les performances
d’un gros LLM (\>1B paramètres) tout en limitant le besoin en puissance
de calcul lors de l’inférence.

Le deuxième avantage d’un si petit modèle est que le fine-tuning ne
nécessite pas un trop grande puissance de calcul, en particulier si des
stratagies de type LORA sont utilisées.

**Question** Explorer le dataset

In [7]:
dataset["train"][0]

{'input': 'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?',
 'output': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.',
 'instruction': 'Answer this question truthfully'}

**Question** Interagir avec le modèle

In [8]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

I'm [Your Name]! I'm a software engineer at [Your Company/Organization]. I'm excited to share my passion for [specific topic or industry]. I'm always looking for ways to

**Question** À la main sur quelques exemples, évaluer la qualité des
réponses du modèle sur ce dataset.

## 3. Fine-tuning

### LoRA: Low Rank Adaptation

On modifie les couches linéaires du modèle

$$
y = W x
$$

où $W \in \mathbb{R}^{d \times k}$ est la matrice de poids originale
(déjà entraînée) et $x$ l’entrée.

Avec LoRA, on ajoute une mise à jour de faible rang $\Delta W = A B$. La
couche linéaire devient:

$$
y = (W + \Delta W) x = (W + A B) x
$$

-   $A \in \mathbb{R}^{d \times r}$, $B \in \mathbb{R}^{r \times k}$
    avec $r \ll \min(d, k)$ (rang faible)
-   Seules les matrices $A$ et $B$ sont entraînables, $W$ reste gelée.
-   Les cibles typiques dans un Transformer sont les matrices de
    projection d’attention: $W_q, W_k, W_v, W_o$.

Avec LoRA, la quasi-totalité des paramètres reste gelée, et on me
modifie qu’un petit nombre de valeurs.

On utilise ici la bibliothèque `PEFT` qui se charge de l’adapation d’un
modèle existant:

In [9]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none"
)

Nombre de paramètres à apprendre:

In [10]:
model_sft = get_peft_model(model, peft_config)
model_sft.print_trainable_parameters()

trainable params: 921,600 || all params: 135,436,608 || trainable%: 0.6805

### SFT: Supervised fine-tuning

On utilise ici la bibliothèque `TRL` qui regroupe toutes les méthodes
pour l’entraînement des LLMs

La fonction suivante contrôle le formatage de ce qui est envoyé au
modèle. On peut travailler dessus pour faire du prompte engineering.

In [11]:
def formatting_prompts_func(example: dict) -> str:
    """Format prompt for training."""
    text = f"<|im_start|>user\n{example['input']}\n{example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"
    return text

In [12]:
sft_config = SFTConfig(
    # output_dir=None,
    num_train_epochs=3,
    max_length=512,
    per_device_train_batch_size=16,
    #gradient_accumulation_steps=2,
    #gradient_checkpointing=False,
    #optim="paged_adamw_32bit",
    save_steps=500,
    logging_steps=500,
    learning_rate=1e-3,
    #weight_decay=0.001,
    fp16=False,
    bf16=False,
    #warmup_ratio=0.05,
    #lr_scheduler_type="constant",
    packing=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_prompts_func,
    peft_config=peft_config,
    args=sft_config,
)

In [13]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=15, training_loss=1.4247491200764975, metrics={'train_runtime': 317.7959, 'train_samples_per_second': 0.755, 'train_steps_per_second': 0.047, 'total_flos': 38071387901952.0, 'train_loss': 1.4247491200764975, 'entropy': 1.335412891705831, 'num_tokens': 30903.0, 'mean_token_accuracy': 0.6822637001673381, 'epoch': 3.0})

**Question** Vérifier la sortie du modèle après cette mise à jour des
poids. Est-ce que la sortie a changé significativement ?

In [ ]:
messages = [
    {"role": "user", "content": "What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(trainer.model.device)

outputs = trainer.model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

## 4. Évaluation

On utilisera la métrice ROUGE qui mesure le chevauchement mot à mot
entre la réponse attendue et la réponse en sortie du modèle. ROUGE
calcule la précision et le rappel mot à mot et renvoie le score F1
(moyenne harmonique de la précision et du rappel).

Précision:

<figure>
<img src="attachment:tp-llm/rouge-precision.png"
alt="ROUGE Precision" />
<figcaption aria-hidden="true">ROUGE Precision</figcaption>
</figure>

Rappel:

<figure>
<img src="attachment:tp-llm/rouge-recall.png" alt="ROUGE Rappel" />
<figcaption aria-hidden="true">ROUGE Rappel</figcaption>
</figure>

Score F1:

$$F_1 = \frac{2}{\mathrm{recall}^{-1} + \mathrm{precision}^{-1}} = 2 \frac{\mathrm{precision} \cdot \mathrm{recall}}{\mathrm{precision} + \mathrm{recall}}$$

**Remarque importante**: il y a plein de métriques d’évaluation pour le
TAL (voire notamment BLEU) mais il s’agit purement d’un score
syntaxique, qui ne s’intéresse pas à la factualité des réponses (ce qui
peut être problématique sur une application médicale comme ici).

In [14]:
rouge = evaluate.load("rouge")

In [15]:
result = rouge.compute(predictions=["Le chat mange les croquettes du chien.", "Il fait beau."], references=["Le chien mange les croquettes du chat.", "Le ciel est bleu."])
result

{'rouge1': np.float64(0.5),
 'rouge2': np.float64(0.25),
 'rougeL': np.float64(0.35714285714285715),
 'rougeLsum': np.float64(0.35714285714285715)}

**Question** Manipuler les phrases d’exemple pour comprendre comment
marche le calcul du score.

On définit une fonction chargée de calculer notre métrique et on
l’ajoute au `trainer`:

In [16]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred

    preds = torch.argmax(torch.Tensor(preds), dim=-1)
    labels = [
        [l if l != -100 else tokenizer.pad_token_id for l in seq]
        for seq in labels
    ]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)# if labels is not None else [""] * len(decoded_preds)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: v for k, v in result.items()}

    return result

La campagne d’évaluation propremet dite sera faite dans la partie
suivante.

## 5. MLops

Lancer le serveur MLFlow avec `uv run mlflow server` et se connecter à
l’interface avec <http://localhost:5000>.

On peut ensuite initialiser le système en donnant à nom à notre série
d’expériences:

In [17]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("SFT_Medical")

<Experiment: artifact_location='file:///mlruns/924866921807019401', creation_time=1764278500431, experiment_id='924866921807019401', last_update_time=1764278500431, lifecycle_stage='active', name='SFT_Medical', tags={'mlflow.experimentKind': 'custom_model_development'}>

On définit ensuite un callback qui sera appelé régulièrement lors de
l’entraînement:

In [18]:
class MLflowStepAndEpochCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        """
        Called at each logging step (controlled by logging_steps in SFTConfig)
        logs: dict containing loss, learning rate, etc.
        """
        if logs:
            # log per-step loss to MLflow
            mlflow.log_metrics({f"loss_step": float(logs.get("loss", 0.0))}, step=state.global_step)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        """
        Called at the end of each evaluation (epoch if evaluation_strategy="epoch")
        """
        if metrics:
            mlflow.log_metrics({k: float(v) for k,v in metrics.items()}, step=int(state.epoch))

Un run se lance de la façon suivante, on peut logguer les
hyperparamètres que l’on souhaite.

In [19]:
with mlflow.start_run():
    sft_config = SFTConfig(
        num_train_epochs=3,
        max_length=512,
        per_device_train_batch_size=16,
        save_steps=1,
        logging_steps=1,
        learning_rate=1e-3,
        fp16=False,
        bf16=False,
        packing=False,
        eval_strategy="epoch"
    )
    trainer = SFTTrainer(
        model=AutoModelForCausalLM.from_pretrained(model_name).to(device),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        formatting_func=formatting_prompts_func,
        peft_config=peft_config,
        args=sft_config,
        compute_metrics=compute_metrics,
    )
    trainer.add_callback(MLflowStepAndEpochCallback())

    mlflow.log_params({
        "model_name": model_name,
        "dataset": dataset_name,
        "num_train_epochs": sft_config.num_train_epochs,
        "per_device_train_batch_size": sft_config.per_device_train_batch_size,
        "learning_rate": sft_config.learning_rate,
        "lora_r": peft_config.r,
        "lora_alpha": peft_config.lora_alpha
    })

    trainer.train()

    mlflow.pytorch.log_model(trainer.model, "sft_cpu_model")

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Entropy,Num Tokens,Mean Token Accuracy
1,1.468400,1.505201,0.691845,0.489274,0.643624,0.670331,1.311231,10301.000000,0.687400
2,1.374800,1.423585,0.710585,0.512977,0.662218,0.691603,1.354353,20602.000000,0.696262
3,1.366300,1.404237,0.711922,0.520098,0.665577,0.690373,1.347476,30903.000000,0.700187


<figure>
<img src="attachment:tp-llm/mlflow_tables.png" alt="MLFlow" />
<figcaption aria-hidden="true">MLFlow</figcaption>
</figure>

<figure>
<img src="attachment:tp-llm/mlflow_figures.png" alt="MLFlow" />
<figcaption aria-hidden="true">MLFlow</figcaption>
</figure>

**Question** Mettre en place une évaluation pour obtenir un tableau de
résultat comparant:

-   le modèle de base
-   le modèle finetuné
-   un gros LLM (type `gpt-5-nano` ou `gpt-4.1-min` avec la clé OpenAI
    fournie)

On rajoutera dans le tableau les différents hyperparamètres explorés.

**Question** Analyser l’influence de la quantité de données utilisée
pour le fine-tuning (en donnant 5%, 10%, etc du dataset)